In [ ]:
import torch
import torchvision 
import matplotlib.pyplot as plt 
import torchvision.transforms as transforms
import torch.utils.data as dataloader


In [3]:
transform = transforms.Compose([transforms.ToTensor()])

In [8]:
from pickle import TRUE
tdata = torchvision.datasets.MNIST(root ='./data',train = True, download = True,transform = transform  )
vdata = torchvision.datasets.MNIST(root ='./data',train = False, download = True,transform = transform  )
batchsize = 64 
imgsize = 10
numclasses = 10
patchsize =7
patchnum = (imgsize/patchsize) * (imgsize/patchsize)
embeddim = 20 
transformerblocks = 4 
mlpnodes = 64
tdata = dataloader.DataLoader(tdata,shuffle = True , batch_size = batchsize)
vdata = dataloader.DataLoader(vdata ,shuffle = True ,batch_size = batchsize)


Embedding

In [16]:
#class for patch embedding 
import torch.nn as nn
class patchembed(nn.Module):
    def __init__(self):
        self.__init__(self)
        self.patchembed = nn.Conv2d(1,embeddim, kernel_size = patchsize, stride = patchsize )
        print("this is the shape of the input image tensor",image.shape)
        print("this is the shape of the imput image tensor after conv2d ",patchembed(image).shape)
    def forward(self,x):
        return self.patchembed(x).flatten(2).transpose(1,2)

In [ ]:
# dimensions analyzer 
images ,labels = next(iter(tdata))
print(images.shape)
patchembed = nn.Conv2d(1,embeddim, kernel_size = patchsize, stride = patchsize )
ourimage = patchembed(images)
print(ourimage.shape)
print(ourimage.flatten(2).shape)

torch.Size([64, 1, 28, 28])
torch.Size([64, 20, 4, 4])
torch.Size([64, 20, 16])


Transformer encoder 

In [18]:
class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layernorm1 = nn.LayerNorm(embeddim)
        self.multiheadattention = nn.MultiheadAttention(embeddim,12)
        self.layernorm2 = nn.LayerNorm(embeddim)
        self.mlp = nn.Sequential(nn.Linear(embeddim),nn.GELU(),nn.Linear(mlpnodes))
    def forward(self,x):
        residual = x
        x = self.layernorm1(x)
        x = self.multiheadattention(x,x,x)[0] + residual 
        residual2 = x
        x = self.layernorm2(x)
        x = self.mlp(x) + residual2
        return x



MLP head 

In [ ]:
class MLPhead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlphead = nn.Sequential(nn.Linear(embeddim),nn.Linear(numclasses))
    def forward(x,self):
        x = x[:,0]  # for classification only the cls token is used 
        x = self.layernorm1(x)
        x = self.mlphead(x)
        return x 


### Programming the vision Transformer 

In [ ]:
class vit(nn.Module):
    def __init__(self):
        self().__init__()
        self.patchembed = patchembed()
        self.cls = nn.Parameter(torch.randn(1,1,embeddim))
        self.position = nn.Parameter(torch.randn(1,patchnum,embeddim))
        self.transformerblock = TransformerEncoder()
        self.mlphead = MLPhead()
    def forward(self,x):
        x = self.patchembed(x)
        b = x.shape[0]
        clstokens = self.cls.expand(b,-1,-1)
        x = torch.cat((clstokens,x),1)
        x = x + self.position
        x = self.transformerblock(x)
        x = self.mlphead(x[:,0])
        x =  self.mlphead(x)
        return x
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
optimizer = torch.optim.Adam(vit(), parameter(),lr = 0.01)
criterion = nn.CrossEntropyLoss()


In [ ]:
torch.randn(1,1,embeddim)

torch.Size([1, 1, 20])